| Task ID | Task Name | Type | Category | Description |
|---------|-----------|------|----------|-------------|
| 4 | Methylation Classification | Binary | Gene Property Prediction | Distinguishes between bivalent and Lys4-only-methylated genes |

In [1]:
import os
import pandas as pd
import pickle
os.chdir(os.path.abspath(".."))
from utils.Gene_level_task_utils import evaluate_embeddings

with open('Dataset/Gene_level_task/bivalent_promoters_bivalent_vs_lys4_only_genomewide.pickle', 'rb') as f:
    data = pickle.load(f)
    
if os.path.exists("Dataset/Gene_level_task/bivalent_promoters_bivalent_vs_lys4_only_genomewide_data.pkl"):
    with open("Dataset/Gene_level_task/bivalent_promoters_bivalent_vs_lys4_only_genomewide_data.pkl", "rb") as f:
        converted_data = pickle.load(f)
    print("load bivalent_promoters_bivalent_vs_lys4_only_genomewide_data.pkl")
else:
    import mygene
    # Initialize the mygene client
    mg = mygene.MyGeneInfo()
    
    # Combine all genes for querying
    all_genes = data['bivalent'] + data['lys4_only']

    # Query mygene for gene symbols
    gene_info = mg.querymany(all_genes, scopes='ensembl.gene', fields='symbol', species='human')

    # Create a mapping from Ensembl IDs to gene symbols
    id_to_symbol = {entry['query']: entry.get('symbol', None) for entry in gene_info}

    # Map Ensembl IDs to gene symbols
    bivalent_symbols = [id_to_symbol.get(ensembl_id) for ensembl_id in data['bivalent'] if id_to_symbol.get(ensembl_id)]
    lys4_only_symbols = [id_to_symbol.get(ensembl_id) for ensembl_id in data['lys4_only'] if id_to_symbol.get(ensembl_id)]

    # Print first few symbols for verification
    print("First 5 gene symbols in 'bivalent':")
    print(bivalent_symbols[:5])

    print("\nFirst 5 gene symbols in 'lys4_only':")
    print(lys4_only_symbols[:5])

    # Prepare the data dictionary
    converted_data = {
        'Bivalent Genes': bivalent_symbols,
        'Lys4 Only Genes': lys4_only_symbols
    }
    with open("Dataset/Gene_level_task/bivalent_promoters_bivalent_vs_lys4_only_genomewide_data.pkl", "wb") as f:
        pickle.dump(converted_data, f)
    print("save bivalent_promoters_bivalent_vs_lys4_only_genomewide_data.pkl")

methods = ["SCGPT-HUMAN","SCGPT-PANCANCER","GF-6L30M","GF-20L95M","GF-12L95M","GF-12L30M", "GF-12L95MCANCER","GENE2VEC","TCGA-EMBEDDING","FROGS-ARCHS4","MUT2VEC","GENEPT-MODEL3","GENEPT-ADA","BIOCONCEPTVEC-SKIP-GRAM","BIOCONCEPTVEC-FASTTEXT","BIOCONCEPTVEC-GLOVE","BIOCONCEPTVEC-CBOW","CoxFormer"]
file_paths = {method: f'Embeddings/{method}.pkl' for method in methods}

# Check if the CSV file exists
csv_file_path = "Result/Gene_level_task/Methylation_Classification.csv"
if os.path.exists(csv_file_path):
    # Load the existing results
    results_df = pd.read_csv(csv_file_path, index_col=0)

    # Check if all methods are already in the results
    existing_methods = results_df['Method'].tolist()
    additional_methods = [method for method in methods if method not in existing_methods]

    if additional_methods:
        print(f"Running evaluation for additional methods: {additional_methods}")

        # Evaluate the additional methods
        new_results_df = evaluate_embeddings(
            data_dict=converted_data,
            positive_label='Bivalent Genes',
            negative_label='Lys4 Only Genes',
            methods=additional_methods,
            file_paths=file_paths
        )

        # Append new results to the existing DataFrame
        results_df = pd.concat([results_df, new_results_df], ignore_index=True)
    else:
        print("All methods are already evaluated. Skipping evaluation.")
else:
    # If the CSV file does not exist, evaluate all methods
    print("CSV file does not exist. Running evaluation for all methods.")
    results_df = evaluate_embeddings(
        data_dict=converted_data,
        positive_label='Bivalent Genes',
        negative_label='Lys4 Only Genes',
        methods=methods,
        file_paths=file_paths
    )

# Save the updated results to CSV
results_df.to_csv(csv_file_path)

load bivalent_promoters_bivalent_vs_lys4_only_genomewide_data.pkl
All methods are already evaluated. Skipping evaluation.
